In [1]:
import pandas as pd
df=pd.read_csv('../data/processed/master_dataset_cleaned.csv',sep=';')
df.head()

,performance_id,interaction_date,campaign_id,customer_id,factor_id,spend,impressions,reach,frequency,clicks,...,Month,Quarter,Year,Day_of_Week,weekend_flag,campaign_duration,budget_utilization,high_competitor_spend_flag,high_discount_flag,YearMonth
0,MP00000001,2025-12-26,CA029,CU00042179,EF726,2808,9206,5166,1.78,242,...,December,4,2025,Friday,No,56,1.138077,Yes,No,2025-12
1,MP00000002,2025-04-13,CA100,CU00023874,EF469,7499,47764,44339,1.08,600,...,April,2,2025,Sunday,Yes,61,1.281068,No,No,2025-04
2,MP00000003,2025-12-17,CA041,CU00022611,EF717,8098,31756,27820,1.14,1003,...,December,4,2025,Wednesday,No,31,2.190780,Yes,No,2025-12
3,MP00000004,2025-02-11,CA097,CU00006368,EF408,1441,5650,3780,1.49,393,...,February,1,2025,Tuesday,No,91,0.857289,No,No,2025-02
4,MP00000005,2025-09-23,CA077,CU00018756,EF632,6101,21184,13012,1.63,913,...,September,3,2025,Tuesday,No,61,1.095826,Yes,No,2025-09


In [2]:
df['interaction_date'] = pd.to_datetime(df['interaction_date'])
df['campaign_start_date'] = pd.to_datetime(df['campaign_start_date'])
df['campaign_end_date'] = pd.to_datetime(df['campaign_end_date'])

In [7]:
df[['spend','revenue','CTR','CVR','ROAS']].describe().loc[['mean','50%','std','min','max']]

,spend,revenue,CTR,CVR,ROAS
mean,6219.409172,182810.324894,0.043183,0.049565,40.313963
50%,4954.000000,126208.000000,0.041634,0.051852,30.661734
std,4446.684399,163925.985728,0.019453,0.022790,37.601615
min,778.000000,2399.000000,0.008328,0.004255,0.324546
max,18651.000000,571872.000000,0.098923,0.114804,347.431349


#### Confidence Interval Estimation

Objective:
Estimate the true population mean of primary KPIs like revenue,ROAS,Profit using 95% confidence intervals.

In [46]:
from scipy import stats
import numpy as np

CI=0.95
alpha=1-CI

mean=df['revenue'].mean()
std=df['revenue'].std()
n=len(df['revenue'])
d_f=n-1

TCV=abs(stats.t.ppf((alpha/2),d_f))
std_err=std/np.sqrt(n)

lower_limit=(mean-(TCV*std_err))
upper_limit=(mean+(TCV*std_err))

print('Estimated average revenue: ',mean)
print(f'Range of True Average revenue is between {round(lower_limit,2)} and {round(upper_limit,2)}')

Estimated average revenue:  182810.324894
Range of True Average revenue is between 182355.95 and 183264.7


**What is the Average revenue?**   
- Estimated Average revenue is 182810.32.  
- Based on Statistical Inference, with 95% Confidence interval the range of average revenue lies b/w 182355.95 and 183264.7

#### HYPOTHESIS TESTING
##### Statistical Assumptions
    - 1 tail test, thus alpha=0.05
    - Welch t-test used because equal variances were not assumed.


**Did the Discount Percentage affect Revenue?**  
Since Discount percentage has more than 2 sample groups with 1 factor, we will use One-way ANOVA

In [54]:
print('Null Hypothesis: Discount Percentage does not affect Revenue')
print('Alternate Hypothesis: Discount Percentage does affect Revenue')
print('-----------------')
df_discount=(df.groupby('discount_percentage')['revenue']).apply(list)

from scipy.stats import f_oneway
f_stats,p_value=f_oneway(*df_discount)
print('Stats:', f_stats)
print('P-value:',p_value)
print('-----------------')
print('Assuming Confidence Interval 95%')
alpha=0.05

if p_value>alpha:
    print('Fail to Reject Null Hypothesis')
    print('Discount Percentage does not affect Revenue')
else:
    print('Reject Null Hypothesis')
    print('Discount Percentage does significantly affect Revenue')

Null Hypothesis: Discount Percentage does not affect Revenue
Alternate Hypothesis: Discount Percentage does affect Revenue
-----------------
Stats: 885.5883666996868
P-value: 0.0
-----------------
Assuming Confidence Interval 95%
Reject Null Hypothesis
Discount Percentage does significantly affect Revenue


**Does Weather affect Revenue?**  
Since Weather has more than 2 sample groups with 1 factor, we will use One-way ANOVA

In [53]:
print('Null Hypothesis: Weather does not affect Revenue')
print('Alternate Hypothesis: Weather does affect Revenue')
print('-----------------')
df_weather=(df.groupby('weather')['revenue']).apply(list)

from scipy.stats import f_oneway
f_stats,p_value=f_oneway(*df_weather)

print('Stats:', f_stats)
print('P-value:',p_value)
print('-----------------')
alpha=0.05
if p_value>alpha:
    print('Fail to Reject Null Hypothesis')
    print('Weather does not affect Revenue')
else:
    print('Reject Null Hypothesis')
    print('Weather does significantly affect Revenue')

Null Hypothesis: Weather does not affect Revenue
Alternate Hypothesis: Weather does affect Revenue
-----------------
Stats: 2264.059040142149
P-value: 0.0
-----------------
Reject Null Hypothesis
Weather does significantly affect Revenue


**Does Marketing Channels affect Revenue?**  
Since Marketing Channels have more than 2 sample groups with 1 factor, we will use One-way ANOVA

In [55]:
print('Null Hypothesis: Marketing Channels does not affect Revenue')
print('Alternate Hypothesis: Marketing channels affect Revenue')
print('-----------------')
df_channel=(df.groupby('channel_name')['revenue']).apply(list)

from scipy.stats import f_oneway
f_stats,p_value=f_oneway(*df_channel)
print('Stats:', f_stats)
print('P-value:',p_value)
print('-----------------')
alpha=0.05

if p_value>alpha:
    print('Fail to Reject Null Hypothesis')
    print('Channel does not affect Revenue')
else:
    print('Reject Null Hypothesis')
    print('Channel does significantly affect Revenue')

Null Hypothesis: Marketing Channels does not affect Revenue
Alternate Hypothesis: Marketing channels affect Revenue
-----------------
Stats: 1576.2518362035478
P-value: 0.0
-----------------
Reject Null Hypothesis
Channel does significantly affect Revenue


**Weekend vs Weekday Revenue?**  
Since we are trying to determine signficant difference between 2 categorical and 1 numerical, we will use independent t test

In [56]:
print('Null Hypothesis: Weekend/Weekday does not affect Revenue')
print('Alternate Hypothesis: Weekend/Weekday affect Revenue')
print('-----------------')

weekend=df[df['weekend_flag']=='Yes']['revenue']
weekday=df[df['weekend_flag']=='No']['revenue']

from scipy.stats import ttest_ind
t_stats,p_value=ttest_ind(weekend,weekday,equal_var=False)

print('Stats:', t_stats)
print('P-value:',p_value)
print('-----------------')

alpha=0.05
if p_value<alpha:
    print('Reject Null Hypothesis')
    print('Weekend revenue does signifcantly affect revenue')
else:
    print('Fail to Reject Null Hypothesis')
    print('Weekend revenue does not affect revenue')

Null Hypothesis: Weekend/Weekday does not affect Revenue
Alternate Hypothesis: Weekend/Weekday affect Revenue
-----------------
Stats: 4.681308160083737
P-value: 2.8519485850067853e-06
-----------------
Reject Null Hypothesis
Weekend revenue does signifcantly affect revenue


**Holiday vs Non-Holiday Revenue**  
Since we are trying to determine signficant difference between 2 categorical and 1 numerical, we will use independent t test

In [57]:
print('Null Hypothesis: Holiday does not affect Revenue')
print('Alternate Hypothesis: Holiday affect Revenue')
print('-----------------')
from scipy.stats import ttest_ind

holiday=df[df['holiday_flag']=='Yes']['revenue']
non_holiday=df[df['holiday_flag']=='No']['revenue']
t_stats,p_value=ttest_ind(holiday,non_holiday,equal_var=False)
print('Stats:', t_stats)
print('P-value:',p_value)
print('-----------------')

if p_value<alpha:
    print('Reject Null Hypothesis')
    print('Holiday revenue does signifcantly affect revenue')
else:
    print('Fail to Reject Null Hypothesis')
    print('Holiday revenue does not affect revenue')

Null Hypothesis: Holiday does not affect Revenue
Alternate Hypothesis: Holiday affect Revenue
-----------------
Stats: 14.968434391803115
P-value: 1.7006600667648218e-50
-----------------
Reject Null Hypothesis
Holiday revenue does signifcantly affect revenue
